# 144 — QLoRA Fine-Tuning

**What you'll learn:**
- 4-bit NF4 quantization and why it drastically cuts VRAM
- The difference between QLoRA and standard LoRA
- How `BitsAndBytesConfig` works parameter by parameter
- What percentage of model parameters are actually trainable with QLoRA
- How to run a full QLoRA training loop with `SFTTrainer`

---
**Source:** `examples/144-qlora-finetuning/`  
**Model:** `HuggingFaceTB/SmolLM2-135M-Instruct`  
**Part A** is CPU-safe. **Part B** requires a GPU (Colab T4 is free).

In [ ]:
%pip install -q transformers peft trl datasets bitsandbytes torch

---
## Part A — CPU-Safe Concept Demonstrations

All cells below run without a GPU and without downloading any model weights.

### A1 — What is 4-bit NF4 Quantization?

Standard neural network weights are stored as `float32` (4 bytes each) or `float16` (2 bytes each).  
**Quantization** maps those weights to a much smaller set of values — NF4 uses only **16 distinct levels**.

#### Memory formula

```
VRAM (bytes) = num_parameters × bytes_per_dtype
```

| dtype | bytes | SmolLM2-135M (135M params) | LLaMA-7B (7B params) |
|-------|-------|---------------------------|----------------------|
| float32 | 4 | ~540 MB | ~28 GB |
| float16 | 2 | ~270 MB | ~14 GB |
| int8 | 1 | ~135 MB | ~7 GB |
| **NF4 (4-bit)** | **0.5** | **~67 MB** | **~3.5 GB** |

#### NF4 vs other 4-bit schemes

NF4 (**N**ormal **F**loat 4) places the 16 quantization levels according to the *normal distribution*.  
Neural network weights tend to follow a bell curve, so NF4 wastes fewer bins on rarely-used extremes compared to uniform int4.

#### QLoRA = Quantize the backbone + train low-rank adapters in higher precision

```
QLoRA pipeline
─────────────────────────────────────────────────────────
  Frozen backbone weights   → stored as NF4 (0.5 bytes/param)
  LoRA adapter matrices A,B → stored as bfloat16 (2 bytes/param)
  Forward pass computation  → promoted to float16 on-the-fly
─────────────────────────────────────────────────────────
Result: 4× smaller base model footprint, full-precision gradients
```

In [ ]:
# A2 — Count trainable parameters (pure Python, no model load)

def count_trainable(mock_params):
    """
    mock_params: list of dicts with keys 'name', 'numel', 'requires_grad'
    Mirrors tools.count_trainable() but works on plain dicts.
    """
    total = sum(p['numel'] for p in mock_params)
    trainable = sum(p['numel'] for p in mock_params if p['requires_grad'])
    pct = 100 * trainable / total
    return {'trainable': trainable, 'total': total, 'pct': round(pct, 4)}


# Simulate a frozen 135M-param backbone + two tiny LoRA adapters
mock_model = [
    # Frozen transformer layers (backbone)
    {'name': 'model.layers.0.self_attn.q_proj.weight', 'numel': 65_536,  'requires_grad': False},
    {'name': 'model.layers.0.self_attn.v_proj.weight', 'numel': 65_536,  'requires_grad': False},
    {'name': 'model.layers.0.mlp.gate_proj.weight',    'numel': 262_144, 'requires_grad': False},
    # LoRA adapter matrices A and B (small, trainable)
    {'name': 'model.layers.0.self_attn.q_proj.lora_A', 'numel': 2_048,   'requires_grad': True},
    {'name': 'model.layers.0.self_attn.q_proj.lora_B', 'numel': 2_048,   'requires_grad': True},
    {'name': 'model.layers.0.self_attn.v_proj.lora_A', 'numel': 2_048,   'requires_grad': True},
    {'name': 'model.layers.0.self_attn.v_proj.lora_B', 'numel': 2_048,   'requires_grad': True},
]

stats = count_trainable(mock_model)

print(f"Trainable params : {stats['trainable']:>10,}")
print(f"Total params     : {stats['total']:>10,}")
print(f"Trainable %      : {stats['pct']:>10.4f}%")
print()
print("In a real QLoRA run on SmolLM2-135M (r=8):")
print("  Trainable params ≈ 720,000  (only ~0.53% of 135M total)")

### A3 — BitsAndBytesConfig Parameters Explained

The `BitsAndBytesConfig` from the `bitsandbytes` library tells HuggingFace how to load the model:

```python
BitsAndBytesConfig(
    load_in_4bit=True,              # (1) Quantize weights to 4 bits on load
    bnb_4bit_quant_type="nf4",      # (2) Use NF4 scheme (vs "fp4")
    bnb_4bit_use_double_quant=True, # (3) Quantize the quantization constants too
    bnb_4bit_compute_dtype=torch.float16,  # (4) Upcast to float16 for matrix multiply
)
```

| Parameter | Values | Effect |
|-----------|--------|--------|
| `load_in_4bit` | `True` / `False` | Enables 4-bit storage |
| `bnb_4bit_quant_type` | `"nf4"` / `"fp4"` | NF4 is better for normally-distributed weights |
| `bnb_4bit_use_double_quant` | `True` / `False` | Saves ~0.4 bits/param extra by quantizing the scale factors |
| `bnb_4bit_compute_dtype` | `torch.float16` / `torch.bfloat16` | Precision used during the actual forward pass |

**Double quantization** applies a second round of quantization to the *scale constants* themselves, squeezing out another ~0.4 bits per parameter. For a 7B model that's roughly 280 MB of additional savings.

In [ ]:
# A4 — Inspect the training data format (no GPU, no model)

def load_sample_dataset(n=50):
    """Mirrors tools.load_sample_dataset() — pure Python, no imports needed."""
    qa_pairs = [
        ("What is gradient descent?",
         "Gradient descent is an optimization algorithm that iteratively adjusts "
         "parameters to minimize a loss function."),
        ("What is backpropagation?",
         "Backpropagation computes gradients of the loss with respect to each weight "
         "using the chain rule."),
        ("What is a transformer?",
         "A transformer is a neural network architecture using self-attention to "
         "process sequences in parallel."),
        ("What is overfitting?",
         "Overfitting occurs when a model learns training data too well, failing to "
         "generalize to new examples."),
        ("What is a learning rate?",
         "The learning rate controls how large a step the optimizer takes toward "
         "the gradient minimum each iteration."),
    ]
    return [{"prompt": q, "response": a} for q, a in qa_pairs * (n // len(qa_pairs) + 1)][:n]


dataset = load_sample_dataset(6)

print(f"Dataset size: {len(dataset)} samples")
print()
for i, row in enumerate(dataset[:3]):
    print(f"--- Sample {i} ---")
    print(f"  prompt  : {row['prompt']}")
    print(f"  response: {row['response'][:60]}...")
    # Show how training text is formatted
    text = f"Q: {row['prompt']}\nA: {row['response']}"
    print(f"  as text : {repr(text[:80])}...")
    print()

# Simulate tokenization shapes without loading a tokenizer
MAX_LEN = 128
sample_text = f"Q: {dataset[0]['prompt']}\nA: {dataset[0]['response']}"
approx_tokens = len(sample_text.split())  # rough word-count estimate
print(f"Approx tokens in sample: ~{approx_tokens}")
print(f"Padded/truncated to max_length={MAX_LEN}")
print(f"Tokenized shape: (batch_size, {MAX_LEN})")

---
## Part B — Full QLoRA Training Run

> **GPU REQUIRED**  
> This section requires a CUDA GPU with at least **4 GB VRAM**.  
> Free option: [Google Colab T4 runtime](https://colab.research.google.com) — Runtime > Change runtime type > T4 GPU.  
> Cells will raise an error on CPU-only machines (that is expected).

In [ ]:
# B1 — GPU check
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected.\n"
        "Part B requires a GPU. On Colab: Runtime > Change runtime type > T4 GPU."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")

In [ ]:
# B2 — Full QLoRA training (from workflow.py)
import json
import sys
sys.path.insert(0, '/content')  # Colab path; adjust if running locally

from src.workflow import create_workflow

workflow = create_workflow()

state = {
    "model_name": "HuggingFaceTB/SmolLM2-135M-Instruct",
    "lora_rank": 8,
    "n_steps": 30,
    "vram_before_mb": 0.0,
    "vram_after_mb": 0.0,
    "param_stats": {},
    "final_loss": 0.0,
}

print(f"Training {state['model_name']} with 4-bit QLoRA (r={state['lora_rank']})...")
result = workflow(state)

print(f"\nVRAM before model load : {result['vram_before_mb']:.0f} MB")
print(f"VRAM after model load  : {result['vram_after_mb']:.0f} MB")
print(f"\nParam stats:\n{json.dumps(result['param_stats'], indent=2)}")
print(f"\nFinal loss: {result['final_loss']}")
print(f"QLoRA keeps {result['param_stats'].get('pct', 0):.4f}% of params trainable.")